In [10]:
!pip install transformers==4.33.3 accelerate==0.21.0 datasets==2.14.0 evaluate==0.4.0


  Using cached transformers-4.33.3-py3-none-any.whl.metadata (119 kB)
  Using cached accelerate-0.21.0-py3-none-any.whl.metadata (17 kB)
  Using cached datasets-2.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached evaluate-0.4.0-py3-none-any.whl.metadata (9.4 kB)
  Using cached tokenizers-0.13.3.tar.gz (314 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached dill-0.3.7-py3-none-any.whl.metadata (9.9 kB)
  Using cached responses-0.18.0-py3-none-any.whl.metadata (29 kB)
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
  Using cached multiprocess-0.70.18-py312-none-any.whl.metadata (7.5 kB)
  Using cached multiprocess-0.70.17-py312-none-any.whl.metadata (7.2 kB)
  Using cached multiprocess-0.70.15-py311-none-any.whl.metadata (7.2 kB)
Using cached transformers-4.33.3-py3-none-any.whl (7.6

In [11]:
#imports
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score

#device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
%cd /content/drive/MyDrive/lm-zoo/
!ls

/content/drive/MyDrive/lm-zoo
glue_data


In [14]:
train_path = "glue_data/SST-2/train.tsv"
dev_path = "glue_data/SST-2/dev.tsv"

train_df = pd.read_csv(train_path, sep="\t")
val_df   = pd.read_csv(dev_path, sep="\t")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
train_df.head()

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df)
})

dataset


Train shape: (67349, 2)
Validation shape: (872, 2)


DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label'],
        num_rows: 872
    })
})

In [15]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(examples):
    return tokenizer(
        examples["sentence"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

encoded = dataset.map(preprocess, batched=True)
encoded

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'input_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'input_ids', 'attention_mask'],
        num_rows: 872
    })
})

In [23]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

training_args = TrainingArguments(
    output_dir="./outputs/distilbert-sst2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [24]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

/tmp/ipython-input-3379418544.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.184800,0.298161,0.893349
2,0.091400,0.425807,0.873853
3,0.074900,0.421874,0.893349


TrainOutput(global_step=12630, training_loss=0.14074039918415526, metrics={'train_runtime': 2137.3434, 'train_samples_per_second': 94.532, 'train_steps_per_second': 5.909, 'total_flos': 6691160124062208.0, 'train_loss': 0.14074039918415526, 'epoch': 3.0})

In [26]:
metrics = trainer.evaluate()
print("Final evaluation:", metrics)


Final evaluation: {'eval_loss': 0.29816100001335144, 'eval_accuracy': 0.893348623853211, 'eval_runtime': 2.9601, 'eval_samples_per_second': 294.58, 'eval_steps_per_second': 4.73, 'epoch': 3.0}


In [27]:
trainer.save_model("./distilbert-sst2-model")
tokenizer.save_pretrained("./distilbert-sst2-model")


('./distilbert-sst2-model/tokenizer_config.json',
 './distilbert-sst2-model/special_tokens_map.json',
 './distilbert-sst2-model/vocab.txt',
 './distilbert-sst2-model/added_tokens.json',
 './distilbert-sst2-model/tokenizer.json')